# Self-Healing Feeder — run on Kaggle

Runs the full study (`run.py`) on Kaggle's CPU so nothing is stored on your own computer.

**Before you start (right-hand panel → Settings):**
1. **Accelerator: None** — the project uses CPU only (OpenDSS + scikit-learn); a GPU would not make it faster and would spend your GPU quota.
2. **Get the code in**, either way:
   - **Upload bundle (no Internet needed):** *Add Input → Upload* the file `self-healing-feeder-kaggle.zip` as a private dataset. It contains the code plus the OpenDSS Linux wheels. Leave `REPO_URL` as it is.
   - **GitHub:** turn **Internet: On** (Kaggle requires a phone-verified account) and set `REPO_URL` to your public repo.

**Then:** **Run All**, or **Save Version → Save & Run All (Commit)** to run in the background (you can close the browser). Results appear in this notebook and in the **Output** panel (`results/` folder) when it finishes.

Full run: 11,448 simulated faults, about 10–20 minutes on Kaggle's ~4 CPU cores. Set `QUICK = True` for a ~1/3-size check run.

In [ ]:
# ---- Settings ----
REPO_URL = "https://github.com/<your-username>/self-healing-feeder.git"  # your PUBLIC GitHub repo

# Alternative without GitHub: upload the project folder as a Kaggle Dataset
# (Add Input -> Upload) and leave REPO_URL as the placeholder above.
DATASET_DIR = "/kaggle/input"

QUICK = False      # True = 1 random load scenario per fault instead of 3 (faster check)
RUN_TESTS = True   # run the 15 automated tests before the study

In [ ]:
# ---- Get the code into /kaggle/working (the only writable, saved folder) ----
import os, shutil, subprocess

WORK = "/kaggle/working/self-healing-feeder"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

if "<your-username>" not in REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
else:
    # find the uploaded project: the folder that contains run.py
    src = next((root for root, _dirs, files in os.walk(DATASET_DIR) if "run.py" in files), None)
    if src is None:
        # the dataset may hold the bundle as a .zip that Kaggle did not unpack
        import zipfile
        zips = [os.path.join(r, f) for r, _d, fs in os.walk(DATASET_DIR) for f in fs if f.endswith(".zip")]
        for z in zips:
            zipfile.ZipFile(z).extractall("/kaggle/working/_bundle")
        src = next((root for root, _dirs, files in os.walk("/kaggle/working/_bundle")
                    if "run.py" in files), None)
    if src is None:
        raise SystemExit("Set REPO_URL to your GitHub repo, or attach the project folder as a Kaggle Dataset.")
    shutil.copytree(src, WORK)   # /kaggle/input is read-only, so work on a copy
    shutil.rmtree("/kaggle/working/_bundle", ignore_errors=True)

os.chdir(WORK)
print("Code in", WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# ---- Install the OpenDSS engine (numpy, pandas, scikit-learn, matplotlib, networkx are preinstalled) ----
# The Kaggle upload bundle ships the Linux wheels in wheels/, so no Internet is needed.
if os.path.isdir(os.path.join(WORK, "wheels")):
    get_ipython().run_line_magic("pip", "install -q --no-index --find-links wheels OpenDSSDirect.py==0.9.4")
else:
    get_ipython().run_line_magic("pip", "install -q OpenDSSDirect.py==0.9.4")

In [ ]:
import platform, numpy, pandas, sklearn, matplotlib, networkx
import opendssdirect as dss

print("Python", platform.python_version(), "| CPU cores:", os.cpu_count())
print("numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scikit-learn", sklearn.__version__, "| matplotlib", matplotlib.__version__,
      "| networkx", networkx.__version__)
print(dss.Basic.Version())

In [ ]:
# ---- Smoke test: the IEEE 123-bus base case should draw about 3,615 kW from the source ----
import sys
sys.path.insert(0, WORK)
from shf import feeder
feeder.load_feeder()
print("source kW:", round(-dss.Circuit.TotalPower()[0], 1))

In [ ]:
# ---- Automated tests (about 15-30 s) ----
if RUN_TESTS:
    get_ipython().system("python -m pytest")

In [ ]:
# ---- Full study: fault simulations, baseline, ML cross-validation, restoration ----
get_ipython().system("python -u run.py" + (" --quick" if QUICK else ""))

In [ ]:
# ---- Results ----
import json
from IPython.display import Image, display

m = json.load(open("results/metrics.json"))
fl = m["fault_location"]
print(f"Faults simulated: {m['dataset']['rows']:,}")
print(f"Accuracy  FPI baseline {fl['fpi_baseline']['accuracy']:.1%} | "
      f"ML magnitude-only {fl['ml_magnitude_only']['accuracy']:.1%} | "
      f"ML full {fl['ml_full']['accuracy']:.1%}")
print(f"Average patrol: {m['patrol_kft']['no_location']:.1f} kft without location -> "
      f"{m['patrol_kft']['ml_full']:.1f} kft with ML zone")
print("Restoration within limits:", m["restoration"]["all_final_states_within_limits"],
      "| radial:", m["restoration"]["all_radial"])

for f in ["feeder_zones.png", "accuracy_vs_resistance.png", "confusion_ml.png"]:
    display(Image(filename=f"results/figures/{f}"))
display(pandas.read_csv("results/restoration.csv"))

In [ ]:
# ---- Put the results at the top of the Output panel (no zip needed) ----
OUT = "/kaggle/working/results"
os.chdir("/kaggle/working")
if os.path.exists(OUT):
    shutil.rmtree(OUT)
shutil.move(os.path.join(WORK, "results"), OUT)
# dataset and model live outside results/ in the repo; keep them in the download too
shutil.copy(os.path.join(WORK, "data", "generated", "dataset.csv.gz"), OUT)
shutil.copy(os.path.join(WORK, "models", "model.pkl"), OUT)
shutil.rmtree(WORK)   # drop the code copy so Output only shows results
for root, _dirs, files in os.walk(OUT):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"{os.path.relpath(p, OUT):40s} {os.path.getsize(p) / 1e3:10.1f} kB")

## Where to see the results

- **On this page:** the *Results* cell above prints the key numbers and shows the three figures and the restoration table.
- **Output panel / Output tab:** the `results/` folder holds every file. Click a file to preview it: PNG figures, `metrics.json`, `restoration.csv`, `zones.csv`, `switching_orders.md`. Download any single file with its download icon.
- **Command line (optional):** `kaggle kernels output <your-username>/<notebook-slug> -p results_kaggle` downloads everything. It needs the Kaggle CLI to be logged in (`kaggle auth login`).